In [ ]:
import os
import json
import yaml
import asyncio
import pandas as pd
import uuid  # 세션 ID 생성을 위해 추가
from tqdm import tqdm
from langchain_openai import ChatOpenAI
from openai import AsyncOpenAI

# 사용자 모듈 (경로 유지)
from rag_model import initialize_rag_pipeline
from utils import rag_utils_es

# 1. 설정 로드
with open('config/config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

JUDGE_MODEL = "gpt-4o"
RAG_MODEL_NAME = config["openai"]["models"]["kgs-answer"]
ES_INDEX = config['elasticsearch']['kgs_index']

# OpenAI 비동기 클라이언트
judge_client = AsyncOpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# 2. MT Bench 스타일 Prompt 정의 (Single Turn, Reference-based)
MTBENCH_REF_PROMPT = """
### Role
You are an expert evaluator for a Question Answering System.
Your task is to rate the quality of the AI generated answer based on the provided "Ground Truth Answer".

### Input
1. **User Question**: {question}
2. **Ground Truth**: {ground_truth}
3. **AI Answer**: {prediction}

### Evaluation Criteria
Please rate the AI Answer on a scale of 1 to 5:
- **1 (Bad)**: The answer is completely incorrect or irrelevant.
- **2 (Poor)**: The answer misses key facts or contains hallucinations.
- **3 (Fair)**: The answer is partially correct but misses some details or is vague.
- **4 (Good)**: The answer is mostly correct and covers the main points.
- **5 (Excellent)**: The answer is accurate, complete, and matches the Ground Truth perfectly in meaning.

### Output Format
You MUST output a JSON object with the following format:
{{
    "score": <integer between 1 and 5>,
    "reasoning": "<short explanation in Korean>"
}}
"""
JUDGE_PROMPT = MTBENCH_REF_PROMPT 

# 3. 동시성 제어 설정
# RAG가 DB/검색 부하가 더 크므로 조금 더 보수적으로 잡음
RAG_CONCURRENCY = 10     
JUDGE_CONCURRENCY = 20   

rag_sem = asyncio.Semaphore(RAG_CONCURRENCY)
judge_sem = asyncio.Semaphore(JUDGE_CONCURRENCY)

def extract_answer(rag_output):
    """RAG 체인 출력에서 실제 답변 텍스트만 추출"""
    if isinstance(rag_output, str):
        return rag_output
    if isinstance(rag_output, dict):
        return rag_output.get("answer") or rag_output.get("output") or rag_output.get("result") or str(rag_output)
    return str(rag_output)

async def evaluate_with_judge(question, ground_truth, prediction):
    prompt = JUDGE_PROMPT.format(
        question=question,
        ground_truth=ground_truth,
        prediction=prediction
    )
    
    async with judge_sem:
        try:
            resp = await judge_client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                response_format={"type": "json_object"}
            )
            
            content = resp.choices[0].message.content.strip()
            
            # [보완] Markdown Code Block 제거 (```json ... ```)
            if content.startswith("```json"):
                content = content[7:]
            if content.endswith("```"):
                content = content[:-3]
            
            result = json.loads(content)
            return int(result.get("score", 0)), result.get("reasoning", "")
            
        except Exception as e:
            return 0, f"Error: {str(e)}"

async def process_one(hit, rag_chain):
    source = hit["_source"]
    
    # 데이터 필드 추출 (이전 대화에서 확인한 구조 반영)
    ground_truth_text = source.get("text")
    metadata = source.get("metadata", {}) or {}
    question_text = metadata.get("question")

    if not question_text or not ground_truth_text:
        return None

    # [중요] 세션 ID 유니크하게 생성 (히스토리 충돌 방지)
    unique_session_id = f"eval_{uuid.uuid4().hex[:8]}"

    # 1) RAG 답변 생성
    async with rag_sem:
        try:
            # config에 session_id를 동적으로 전달
            rag_output = await rag_chain.ainvoke(
                {"input": question_text, "summary": "이전 대화 없음", "chat_history_8": []},
                config={"configurable": {"session_id": unique_session_id}}
            )
            ai_answer = extract_answer(rag_output)
        except Exception as e:
            ai_answer = f"Error generating answer: {e}"

    # 2) Judge 평가
    score, reasoning = await evaluate_with_judge(question_text, ground_truth_text, ai_answer)

    return {
        "question": question_text,
        "ground_truth": ground_truth_text,
        "ai_answer": ai_answer,
        "score": score,
        "reasoning": reasoning,
        "category": metadata.get("category1", "unknown"),
    }

async def run_evaluation():
    print(f"🚀 '{ES_INDEX}' 인덱스 RAG 성능 평가 시작 (MT-Bench Style)")

    es_client = rag_utils_es.get_es_client()

    # 1. 데이터 로드
    query = {
        "query": {"match_all": {}},
        "size": 2000, # 전체 데이터 커버
        "_source": ["text", "metadata"]
    }
    response = es_client.search(index=ES_INDEX, body=query)
    hits = response["hits"]["hits"]
    
    if not hits:
        print("🚨 데이터를 찾을 수 없습니다.")
        return

    print(f"✅ 총 {len(hits)}개의 샘플 로드 완료. 평가를 시작합니다...")

    # 2. RAG 체인 로드
    llm_answer = ChatOpenAI(model=RAG_MODEL_NAME, temperature=0)
    rag_chain = initialize_rag_pipeline(config, llm_answer)

    # 3. 비동기 작업 생성
    tasks = [process_one(hit, rag_chain) for hit in hits]

    results = []
    # as_completed를 사용하여 완료되는 순서대로 게이지 바 업데이트
    for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Evaluating"):
        item = await coro
        if item is not None:
            results.append(item)

    # 4. 결과 저장
    df = pd.DataFrame(results)
    avg_score = df["score"].mean() if not df.empty else 0

    print("\n" + "=" * 30)
    print(f"📊 평균 점수: {avg_score:.2f} / 5.0")
    print("=" * 30)

    output_file = "rag_evaluation_mtbench_full.csv"
    df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"📄 결과 파일 저장 완료: {output_file}")

if __name__ == "__main__":
    if os.name == "nt":
        asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
    asyncio.run(run_evaluation())